In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
TriLiver-FuseNet: Multi-Representation Liver Tumor Classification
================================================================

Proposed model for the journal paper:
"Leveraging deep learning and explainable AI for effective liver tumor
classification from CT scan images" (Alfarhood et al., 2026).

Novel idea
----------
The original work evaluates single image backbones. This proposed model fuses:
1. Global CT deep features from ImageNet-pretrained EfficientNetV2-S.
2. Texture features: LBP, intensity histogram and edge histogram.
3. Quantitative radiomics-inspired features: GLCM and statistical descriptors.
4. Adaptive modality attention and gated residual fusion.

The script includes:
- kaggle.json upload and automatic Kaggle dataset download
- corrupted-image filtering
- train/validation/test stratified split
- bilateral filtering, CLAHE and resizing
- train-only augmentation
- class weighting
- two-stage training and fine-tuning
- accuracy, precision, recall/sensitivity, specificity, F1 and AUC
- confusion matrices, ROC curves, metric box plots and bright curves
- before/after preprocessing samples
- model parameters, FLOPs/GFLOPs and training time
- Grad-CAM for the first 20 test images
- modality-ablation explainability
- result ZIP and automatic Colab download

Run in Colab
------------
!python triliver_fusenet_full_pipeline.py
"""

# -----------------------------------------------------------------------------
# 0. DEPENDENCIES
# -----------------------------------------------------------------------------
import os
import sys
import json
import time
import math
import shutil
import random
import zipfile
import warnings
import subprocess
import importlib.util
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")


def install_if_missing(import_name: str, pip_name: Optional[str] = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"[INSTALL] {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


for imp, pkg in [
    ("kaggle", "kaggle"),
    ("cv2", "opencv-python-headless"),
    ("sklearn", "scikit-learn"),
    ("skimage", "scikit-image"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
]:
    install_if_missing(imp, pkg)

# -----------------------------------------------------------------------------
# 1. IMPORTS
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.measure import shannon_entropy

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    auc,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S

# -----------------------------------------------------------------------------
# 2. CONFIGURATION
# -----------------------------------------------------------------------------
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 20
FINE_TUNE_EPOCHS = 30
TOTAL_EPOCHS = HEAD_EPOCHS + FINE_TUNE_EPOCHS
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
TEST_SIZE = 0.20
VAL_SIZE_FROM_REMAINING = 0.20
N_XAI = 20

KAGGLE_KERNEL_REF = "ahmedhamza1996/liver-tumor-classification"
KAGGLE_DATASET_SLUG = ""  # Optional direct slug: owner/dataset-name

USE_BILATERAL_FILTER = True
USE_CLAHE = True

WORK_DIR = (
    Path("/content/liver_triliver_fusenet_work")
    if Path("/content").exists()
    else Path.cwd() / "liver_triliver_fusenet_work"
)
DATA_DIR = WORK_DIR / "dataset"
RESULTS_DIR = WORK_DIR / "results_triliver_fusenet"
MODEL_DIR = RESULTS_DIR / "model"
PLOTS_DIR = RESULTS_DIR / "plots"
METRICS_DIR = RESULTS_DIR / "metrics"
XAI_DIR = RESULTS_DIR / "xai"
SAMPLES_DIR = RESULTS_DIR / "preprocessing_samples"

for folder in [WORK_DIR, DATA_DIR, RESULTS_DIR, MODEL_DIR, PLOTS_DIR, METRICS_DIR, XAI_DIR, SAMPLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CLASS_ALIASES = {
    "normal": ["normal", "healthy", "no_tumor", "no-tumor", "notumor"],
    "benign": ["benign", "cyst", "hemangioma", "hydatid"],
    "malignant": ["malignant", "cancer", "hcc", "metastasis", "tumor"],
}
TARGET_CLASS_ORDER = ["normal", "benign", "malignant"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

BRIGHT_COLORS = {
    "train": "#00BFFF",
    "validation": "#FF1493",
    "test": "#32CD32",
    "loss_train": "#FF8C00",
    "loss_validation": "#9400D3",
    "loss_test": "#00CED1",
}

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

GPUS = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU devices:", GPUS)
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

if GPUS:
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision:", mixed_precision.global_policy())
    except Exception as exc:
        print("Mixed precision disabled:", exc)

# -----------------------------------------------------------------------------
# 3. KAGGLE DOWNLOAD
# -----------------------------------------------------------------------------
def in_colab() -> bool:
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False


def configure_kaggle_credentials() -> None:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / "kaggle.json"

    if target.exists():
        os.chmod(target, 0o600)
        print("Kaggle credentials found:", target)
        return

    for candidate in [Path.cwd() / "kaggle.json", Path("/content/kaggle.json"), WORK_DIR / "kaggle.json"]:
        if candidate.exists():
            shutil.copy2(candidate, target)
            os.chmod(target, 0o600)
            print("Configured Kaggle credentials from:", candidate)
            return

    if in_colab():
        from google.colab import files
        print("\nPlease upload kaggle.json...")
        uploaded = files.upload()
        names = [name for name in uploaded if name.lower().endswith(".json")]
        if not names:
            raise FileNotFoundError("No kaggle.json uploaded.")
        shutil.copy2(Path(names[0]), target)
        os.chmod(target, 0o600)
    else:
        raise FileNotFoundError("Place kaggle.json in the current directory or ~/.kaggle/.")


def run_command(command: List[str], check: bool = True) -> subprocess.CompletedProcess:
    print("[CMD]", " ".join(command))
    return subprocess.run(command, check=check, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)


def parse_kernel_dataset_sources(metadata_dir: Path) -> List[str]:
    files = list(metadata_dir.rglob("*metadata*.json")) + list(metadata_dir.rglob("kernel-metadata.json"))
    slugs: List[str] = []
    for file in files:
        try:
            data = json.loads(file.read_text(encoding="utf-8"))
        except Exception:
            continue
        for key in ["dataset_sources", "datasetSources"]:
            values = data.get(key, [])
            if isinstance(values, list):
                for item in values:
                    if isinstance(item, str) and "/" in item:
                        slugs.append(item)
                    elif isinstance(item, dict):
                        ref = item.get("ref") or item.get("source") or item.get("dataset")
                        if isinstance(ref, str) and "/" in ref:
                            slugs.append(ref)
        values = data.get("data_sources", [])
        if isinstance(values, list):
            for item in values:
                if isinstance(item, dict):
                    ref = item.get("ref") or item.get("source")
                    typ = str(item.get("sourceType", item.get("type", ""))).lower()
                    if isinstance(ref, str) and "/" in ref and ("dataset" in typ or not typ):
                        slugs.append(ref)
    return sorted(set(slugs))


def discover_kernel_inputs(kernel_ref: str) -> List[str]:
    metadata_dir = WORK_DIR / "kaggle_kernel_metadata"
    if metadata_dir.exists():
        shutil.rmtree(metadata_dir)
    metadata_dir.mkdir(parents=True, exist_ok=True)

    commands = [
        ["kaggle", "kernels", "pull", kernel_ref, "-p", str(metadata_dir), "-m"],
        ["kaggle", "kernels", "pull", "-p", str(metadata_dir), "-m", kernel_ref],
    ]
    for command in commands:
        try:
            result = run_command(command)
            print(result.stdout[-1200:])
            slugs = parse_kernel_dataset_sources(metadata_dir)
            print("Discovered Kaggle dataset sources:", slugs)
            return slugs
        except Exception:
            continue
    return []


def download_data() -> List[str]:
    configure_kaggle_credentials()
    slugs = [KAGGLE_DATASET_SLUG.strip()] if KAGGLE_DATASET_SLUG.strip() else discover_kernel_inputs(KAGGLE_KERNEL_REF)
    if not slugs:
        raise RuntimeError("No Kaggle dataset slug detected. Set KAGGLE_DATASET_SLUG manually.")

    for slug in slugs:
        destination = DATA_DIR / slug.replace("/", "__")
        destination.mkdir(parents=True, exist_ok=True)
        marker = destination / ".download_complete"
        if marker.exists() and any(destination.rglob("*")):
            print("Dataset already downloaded:", slug)
            continue
        result = run_command(["kaggle", "datasets", "download", "-d", slug, "-p", str(destination), "--unzip"])
        print(result.stdout[-1500:])
        marker.touch()
    return slugs

# -----------------------------------------------------------------------------
# 4. DATASET DISCOVERY
# -----------------------------------------------------------------------------
def normalize_token(text: str) -> str:
    return text.lower().replace(" ", "_").replace("-", "_")


def infer_label(path: Path) -> Optional[str]:
    components = [normalize_token(part) for part in path.parts]
    for label in TARGET_CLASS_ORDER:
        aliases = [normalize_token(x) for x in CLASS_ALIASES[label]]
        for component in reversed(components[:-1]):
            if component == label or component in aliases:
                return label
    joined = "/".join(components)
    for label in ["malignant", "benign", "normal"]:
        for alias in CLASS_ALIASES[label]:
            if normalize_token(alias) in joined:
                return label
    return None


def read_bgr_robust(path: str) -> Optional[np.ndarray]:
    try:
        raw = np.fromfile(path, dtype=np.uint8)
        if raw.size == 0:
            return None
        return cv2.imdecode(raw, cv2.IMREAD_COLOR)
    except Exception:
        return None


def collect_images(root: Path) -> pd.DataFrame:
    valid_rows, bad_rows = [], []
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        label = infer_label(path)
        if label is None:
            continue
        image = read_bgr_robust(str(path))
        if image is None or image.ndim != 3 or min(image.shape[:2]) < 8:
            bad_rows.append({"filepath": str(path), "label": label, "reason": "decode_or_shape_failed"})
        else:
            valid_rows.append({"filepath": str(path), "label": label})

    pd.DataFrame(bad_rows, columns=["filepath", "label", "reason"]).to_csv(
        METRICS_DIR / "skipped_corrupt_images.csv", index=False
    )
    df = pd.DataFrame(valid_rows).drop_duplicates("filepath")
    if df.empty:
        raise RuntimeError("No valid labelled images found.")
    print("\nDetected classes:\n", df["label"].value_counts())
    return df.reset_index(drop=True)


def create_splits(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_val, test = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df["label"])
    train, val = train_test_split(
        train_val,
        test_size=VAL_SIZE_FROM_REMAINING,
        random_state=SEED,
        stratify=train_val["label"],
    )
    for name, split in [("training", train), ("validation", val), ("test", test)]:
        split.to_csv(METRICS_DIR / f"{name}_split.csv", index=False)
        print(f"\n{name.upper()} ({len(split)})\n", split["label"].value_counts())
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

# -----------------------------------------------------------------------------
# 5. PREPROCESSING AND HANDCRAFTED MODALITIES
# -----------------------------------------------------------------------------
def read_rgb(path: str) -> np.ndarray:
    bgr = read_bgr_robust(path)
    if bgr is None:
        raise ValueError(f"Cannot decode image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def preprocess_rgb(image_rgb: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(np.asarray(image_rgb, dtype=np.uint8), cv2.COLOR_RGB2GRAY)
    if USE_BILATERAL_FILTER:
        gray = cv2.bilateralFilter(gray, 9, 75, 75)
    if USE_CLAHE:
        gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)


def texture_features(image_rgb: np.ndarray) -> np.ndarray:
    processed = preprocess_rgb(image_rgb)
    gray = cv2.cvtColor(processed, cv2.COLOR_RGB2GRAY)

    radius = 2
    points = 8 * radius
    lbp = local_binary_pattern(gray, points, radius, method="uniform")
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, points + 3), range=(0, points + 2), density=True)

    intensity = cv2.calcHist([gray], [0], None, [32], [0, 256]).ravel()
    intensity = intensity / (intensity.sum() + 1e-8)

    sx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(sx, sy)
    edge_hist, _ = np.histogram(magnitude.ravel(), bins=32, range=(0, max(float(magnitude.max()), 1.0)), density=True)

    return np.nan_to_num(np.concatenate([lbp_hist, intensity, edge_hist])).astype(np.float32)


def quantitative_features(image_rgb: np.ndarray) -> np.ndarray:
    processed = preprocess_rgb(image_rgb)
    gray = cv2.cvtColor(processed, cv2.COLOR_RGB2GRAY)
    gray_q = (gray // 16).astype(np.uint8)  # 16 gray levels

    glcm = graycomatrix(gray_q, distances=[1, 2], angles=[0, np.pi / 4, np.pi / 2], levels=16, symmetric=True, normed=True)
    props = []
    for name in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        props.extend(graycoprops(glcm, name).ravel().tolist())

    percentiles = np.percentile(gray, [5, 10, 25, 50, 75, 90, 95])
    sx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(sx, sy)

    stats = [
        float(gray.mean()), float(gray.std()), float(gray.min()), float(gray.max()),
        float(shannon_entropy(gray)), *percentiles.tolist(),
        float(np.mean(np.abs(sx))), float(np.mean(np.abs(sy))), float(np.mean(mag)), float(np.std(mag)),
    ]
    return np.nan_to_num(np.asarray(props + stats, dtype=np.float32))


def save_preprocessing_samples(df: pd.DataFrame, n: int = 8) -> None:
    sample = df.sample(min(n, len(df)), random_state=SEED)
    fig, axes = plt.subplots(len(sample), 2, figsize=(8, max(3 * len(sample), 6)))
    if len(sample) == 1:
        axes = np.array([axes])
    for i, (_, row) in enumerate(sample.iterrows()):
        original = read_rgb(row.filepath)
        processed = preprocess_rgb(original)
        axes[i, 0].imshow(original)
        axes[i, 0].set_title(f"Before: {row.label}", fontweight="bold")
        axes[i, 1].imshow(processed)
        axes[i, 1].set_title(f"After: {row.label}", fontweight="bold")
        axes[i, 0].axis("off")
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.savefig(SAMPLES_DIR / "before_after_preprocessing.png", dpi=300, bbox_inches="tight")
    plt.close()


def extract_feature_arrays(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    textures, quantitative = [], []
    for i, path in enumerate(df.filepath, start=1):
        if i == 1 or i % 100 == 0:
            print(f"Handcrafted feature extraction: {i}/{len(df)}")
        image = read_rgb(path)
        textures.append(texture_features(image))
        quantitative.append(quantitative_features(image))
    return np.asarray(textures, np.float32), np.asarray(quantitative, np.float32)

# -----------------------------------------------------------------------------
# 6. TF.DATA
# -----------------------------------------------------------------------------
def load_image_numpy(path_value: Any) -> np.ndarray:
    if isinstance(path_value, np.ndarray):
        path_value = path_value.item()
    if isinstance(path_value, (bytes, np.bytes_)):
        path_value = path_value.decode("utf-8")
    image = read_rgb(str(path_value))
    return preprocess_rgb(image).astype(np.float32)


def load_image_tf(path: tf.Tensor) -> tf.Tensor:
    image = tf.numpy_function(load_image_numpy, [path], tf.float32)
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return image


def augmentation_layer() -> keras.Sequential:
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        layers.RandomRotation(20.0 / 360.0, fill_mode="reflect", seed=SEED),
        layers.RandomTranslation(0.06, 0.06, fill_mode="reflect", seed=SEED),
        layers.RandomZoom((-0.10, 0.10), (-0.10, 0.10), seed=SEED),
        layers.RandomContrast(0.10, seed=SEED),
    ], name="training_augmentation")


def make_dataset(
    df: pd.DataFrame,
    texture_array: np.ndarray,
    quantitative_array: np.ndarray,
    class_to_index: Dict[str, int],
    training: bool,
) -> tf.data.Dataset:
    paths = df.filepath.astype(str).values
    labels = df.label.map(class_to_index).astype(np.int32).values

    ds = tf.data.Dataset.from_tensor_slices((paths, texture_array, quantitative_array, labels))
    if training:
        ds = ds.shuffle(max(len(df), 1), seed=SEED, reshuffle_each_iteration=True)
    aug = augmentation_layer()

    def mapper(path, texture, quantitative, label):
        image = load_image_tf(path)
        if training:
            image = aug(image, training=True)
        return {
            "image_input": image,
            "texture_input": tf.cast(texture, tf.float32),
            "quantitative_input": tf.cast(quantitative, tf.float32),
        }, tf.cast(label, tf.int32)

    ds = ds.map(mapper, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# -----------------------------------------------------------------------------
# 7. PROPOSED MODEL
# -----------------------------------------------------------------------------
def build_triliver_fusenet(num_classes: int, texture_dim: int, quantitative_dim: int) -> Tuple[keras.Model, keras.Model]:
    image_input = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image_input")
    texture_input = keras.Input((texture_dim,), name="texture_input")
    quantitative_input = keras.Input((quantitative_dim,), name="quantitative_input")

    backbone = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_tensor=image_input,
        pooling=None,
        include_preprocessing=True,
    )
    backbone.trainable = False

    image_feature_map = backbone.output
    image_vector = layers.GlobalAveragePooling2D(name="image_gap")(image_feature_map)
    image_vector = layers.Dense(256, activation="swish", name="image_projection")(image_vector)
    image_vector = layers.BatchNormalization(name="image_bn")(image_vector)
    image_vector = layers.Dropout(0.30, name="image_dropout")(image_vector)

    texture_vector = layers.BatchNormalization(name="texture_input_bn")(texture_input)
    texture_vector = layers.Dense(128, activation="swish", name="texture_projection_1")(texture_vector)
    texture_vector = layers.Dropout(0.20, name="texture_dropout")(texture_vector)
    texture_vector = layers.Dense(128, activation="swish", name="texture_projection_2")(texture_vector)

    quantitative_vector = layers.BatchNormalization(name="quantitative_input_bn")(quantitative_input)
    quantitative_vector = layers.Dense(96, activation="swish", name="quantitative_projection_1")(quantitative_vector)
    quantitative_vector = layers.Dropout(0.20, name="quantitative_dropout")(quantitative_vector)
    quantitative_vector = layers.Dense(96, activation="swish", name="quantitative_projection_2")(quantitative_vector)

    # All modalities projected to a common 256-D space.
    image_common = layers.Dense(256, activation=None, name="image_common")(image_vector)
    texture_common = layers.Dense(256, activation=None, name="texture_common")(texture_vector)
    quantitative_common = layers.Dense(256, activation=None, name="quantitative_common")(quantitative_vector)

    # Use only serializable built-in Keras layers. Avoid Lambda layers because
    # Keras 3 may fail to infer their output shape when reloading a .keras model.
    image_token = layers.Reshape((1, 256), name="image_modality_token")(image_common)
    texture_token = layers.Reshape((1, 256), name="texture_modality_token")(texture_common)
    quantitative_token = layers.Reshape((1, 256), name="quantitative_modality_token")(quantitative_common)
    stacked = layers.Concatenate(axis=1, name="modality_stack")([
        image_token, texture_token, quantitative_token
    ])

    # Adaptive modality attention: one normalized importance score per modality.
    modality_scores = layers.Dense(64, activation="tanh", name="modality_score_hidden")(stacked)
    modality_scores = layers.Dense(1, activation=None, name="modality_score_logits")(modality_scores)
    modality_weights = layers.Softmax(axis=1, name="modality_attention_weights")(modality_scores)
    weighted_modalities = layers.Multiply(name="weighted_modalities")([stacked, modality_weights])

    # GlobalAveragePooling1D divides by the three modality tokens. Since the
    # attention weights already sum to one, multiply the average by three to
    # recover the weighted sum. Both layers are fully serializable.
    attended = layers.GlobalAveragePooling1D(name="attention_fusion_average")(weighted_modalities)
    attended = layers.Rescaling(scale=3.0, name="attention_fusion")(attended)

    concatenated = layers.Concatenate(name="raw_multimodal_concat")([
        image_vector, texture_vector, quantitative_vector
    ])
    residual = layers.Dense(256, activation=None, name="residual_projection")(concatenated)

    gate_input = layers.Concatenate(name="gate_input")([attended, residual])
    gate = layers.Dense(256, activation="sigmoid", name="adaptive_fusion_gate")(gate_input)
    inverse_gate = layers.Rescaling(scale=-1.0, offset=1.0, name="inverse_gate")(gate)
    gated_attended = layers.Multiply(name="gated_attended")([gate, attended])
    gated_residual = layers.Multiply(name="gated_residual")([inverse_gate, residual])
    fused = layers.Add(name="gated_residual_fusion")([gated_attended, gated_residual])

    fused = layers.LayerNormalization(name="fusion_layer_norm")(fused)
    refined = layers.Dense(256, activation="swish", name="fusion_refinement_1")(fused)
    refined = layers.Dropout(0.35, name="fusion_dropout")(refined)
    refined = layers.Dense(128, activation="swish", name="fusion_refinement_2")(refined)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="predictions")(refined)

    model = keras.Model(
        inputs=[image_input, texture_input, quantitative_input],
        outputs=outputs,
        name="TriLiver_FuseNet",
    )
    return model, backbone


def compile_model(model: keras.Model, learning_rate: float) -> None:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )


def unfreeze_backbone(backbone: keras.Model, last_n_layers: int = 60) -> None:
    backbone.trainable = True
    for layer in backbone.layers[:-last_n_layers]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

# -----------------------------------------------------------------------------
# 8. METRICS
# -----------------------------------------------------------------------------
def specificity_per_class(cm: np.ndarray) -> np.ndarray:
    total = cm.sum()
    out = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        out.append(tn / (tn + fp) if tn + fp > 0 else np.nan)
    return np.asarray(out, float)


def safe_auc(y_true: np.ndarray, probs: np.ndarray, num_classes: int) -> float:
    try:
        y_bin = label_binarize(y_true, classes=np.arange(num_classes))
        return float(roc_auc_score(y_bin, probs, average="macro", multi_class="ovr"))
    except Exception:
        return float("nan")


def predict_dataset(model: keras.Model, ds: tf.data.Dataset) -> Tuple[np.ndarray, np.ndarray]:
    y_true, probs = [], []
    for inputs, labels in ds:
        probs.append(np.asarray(model.predict_on_batch(inputs)))
        y_true.append(labels.numpy())
    return np.concatenate(y_true), np.concatenate(probs)


def evaluate_split(model: keras.Model, ds: tf.data.Dataset, split: str, class_names: List[str]):
    evaluation = model.evaluate(ds, verbose=0, return_dict=True)
    y_true, probs = predict_dataset(model, ds)
    y_pred = np.argmax(probs, axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(class_names)), zero_division=0
    )
    specificity = specificity_per_class(cm)

    overall = {
        "split": split,
        "loss": float(evaluation["loss"]),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "sensitivity_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "specificity_macro": float(np.nanmean(specificity)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "auc_macro_ovr": safe_auc(y_true, probs, len(class_names)),
        "n_samples": int(len(y_true)),
    }

    per_class = pd.DataFrame({
        "split": split,
        "class": class_names,
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1_score": f1,
        "support": support,
    })

    y_bin = label_binarize(y_true, classes=np.arange(len(class_names)))
    per_class["auc_ovr"] = [
        roc_auc_score(y_bin[:, i], probs[:, i]) if len(np.unique(y_bin[:, i])) > 1 else np.nan
        for i in range(len(class_names))
    ]
    per_class["accuracy_ovr"] = [
        ((cm[i, i] + (cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i])) / cm.sum())
        for i in range(len(class_names))
    ]

    pd.DataFrame([overall]).to_csv(METRICS_DIR / f"{split}_overall_metrics.csv", index=False)
    per_class.to_csv(METRICS_DIR / f"{split}_per_class_metrics.csv", index=False)
    pd.DataFrame(classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)).transpose().to_csv(
        METRICS_DIR / f"{split}_classification_report.csv"
    )

    pred_df = pd.DataFrame({
        "true_index": y_true,
        "predicted_index": y_pred,
        "true_class": [class_names[i] for i in y_true],
        "predicted_class": [class_names[i] for i in y_pred],
        "confidence": probs.max(axis=1),
        "correct": (y_true == y_pred).astype(int),
    })
    for i, name in enumerate(class_names):
        pred_df[f"prob_{name}"] = probs[:, i]
    pred_df.to_csv(METRICS_DIR / f"{split}_predictions.csv", index=False)
    return overall, y_true, y_pred, probs, per_class

# -----------------------------------------------------------------------------
# 9. PLOTS
# -----------------------------------------------------------------------------
def save_confusion(y_true, y_pred, classes, split):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(classes)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="turbo")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(classes)), classes, rotation=35, ha="right")
    ax.set_yticks(range(len(classes)), classes)
    ax.set_xlabel("Predicted label", fontweight="bold")
    ax.set_ylabel("True label", fontweight="bold")
    ax.set_title(f"{split.title()} Confusion Matrix", fontweight="bold")
    threshold = cm.max() / 2 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > threshold else "black", fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()


def save_roc(y_true, probs, classes, split):
    y_bin = label_binarize(y_true, classes=np.arange(len(classes)))
    fig, ax = plt.subplots(figsize=(8, 7))
    colors = plt.cm.hsv(np.linspace(0, 0.85, len(classes)))
    for i, (name, color) in enumerate(zip(classes, colors)):
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
            ax.plot(fpr, tpr, linewidth=2.5, color=color, label=f"{name} (AUC={auc(fpr, tpr):.3f})")
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], "--", color="black")
    ax.set_xlabel("False Positive Rate", fontweight="bold")
    ax.set_ylabel("True Positive Rate", fontweight="bold")
    ax.set_title(f"{split.title()} ROC-AUC", fontweight="bold")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split}_roc_auc.png", dpi=300, bbox_inches="tight")
    plt.close()


def save_boxplot(per_class, split):
    cols = ["accuracy_ovr", "precision", "recall_sensitivity", "specificity", "f1_score", "auc_ovr"]
    labels = ["Accuracy", "Precision", "Recall/\nSensitivity", "Specificity", "F1", "AUC"]
    fig, ax = plt.subplots(figsize=(10, 6))
    boxes = ax.boxplot([per_class[c].dropna().values for c in cols], labels=labels, patch_artist=True, showmeans=True)
    colors = plt.cm.hsv(np.linspace(0, 0.85, len(boxes["boxes"])))
    for box, color in zip(boxes["boxes"], colors):
        box.set_facecolor(color)
        box.set_alpha(0.65)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Metric value", fontweight="bold")
    ax.set_title(f"{split.title()} Per-Class Metric Box Plot", fontweight="bold")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{split}_metric_boxplot.png", dpi=300, bbox_inches="tight")
    plt.close()


def merge_histories(h1, h2):
    keys = set(h1.history) | set(h2.history)
    return {key: list(h1.history.get(key, [])) + list(h2.history.get(key, [])) for key in keys}


def save_curves(history):
    epochs = np.arange(1, len(history["accuracy"]) + 1)
    pd.DataFrame({"epoch": epochs, **history}).to_csv(METRICS_DIR / "training_history.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(epochs, history["accuracy"], color=BRIGHT_COLORS["train"], linewidth=2.5, label="Training")
    ax.plot(epochs, history["val_accuracy"], color=BRIGHT_COLORS["validation"], linewidth=2.5, label="Validation")
    ax.axvline(HEAD_EPOCHS, color="black", linestyle="--", label="Fine-tuning")
    ax.set_xlabel("Epoch", fontweight="bold")
    ax.set_ylabel("Accuracy", fontweight="bold")
    ax.set_title("TriLiver-FuseNet Accuracy Curves", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.savefig(PLOTS_DIR / "accuracy_curves.png", dpi=300, bbox_inches="tight"); plt.close()

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(epochs, history["loss"], color=BRIGHT_COLORS["loss_train"], linewidth=2.5, label="Training")
    ax.plot(epochs, history["val_loss"], color=BRIGHT_COLORS["loss_validation"], linewidth=2.5, label="Validation")
    ax.axvline(HEAD_EPOCHS, color="black", linestyle="--", label="Fine-tuning")
    ax.set_xlabel("Epoch", fontweight="bold")
    ax.set_ylabel("Loss", fontweight="bold")
    ax.set_title("TriLiver-FuseNet Loss Curves", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.savefig(PLOTS_DIR / "loss_curves.png", dpi=300, bbox_inches="tight"); plt.close()

# -----------------------------------------------------------------------------
# 10. COMPLEXITY
# -----------------------------------------------------------------------------
def count_params(model):
    return {
        "total_parameters": int(model.count_params()),
        "trainable_parameters": int(sum(np.prod(w.shape) for w in model.trainable_weights)),
        "non_trainable_parameters": int(sum(np.prod(w.shape) for w in model.non_trainable_weights)),
    }


def estimate_flops(model):
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        specs = [
            tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32),
            tf.TensorSpec([1, model.inputs[1].shape[-1]], tf.float32),
            tf.TensorSpec([1, model.inputs[2].shape[-1]], tf.float32),
        ]
        concrete = tf.function(lambda a, b, c: model([a, b, c], training=False)).get_concrete_function(*specs)
        frozen = convert_variables_to_constants_v2(concrete)
        with tf.Graph().as_default() as graph:
            tf.compat.v1.graph_util.import_graph_def(frozen.graph.as_graph_def(), name="")
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            profile = tf.compat.v1.profiler.profile(graph=graph, cmd="op", options=opts)
            flops = int(profile.total_float_ops) if profile else None
        return flops, flops / 1e9 if flops else None
    except Exception as exc:
        print("FLOPs estimation failed:", exc)
        return None, None

# -----------------------------------------------------------------------------
# 11. XAI
# -----------------------------------------------------------------------------
def normalize_heatmap(h):
    h = np.squeeze(np.asarray(h, dtype=np.float32))
    h = np.nan_to_num(h)
    h = np.maximum(h, 0)
    return h / (h.max() + 1e-8)


def gradcam(model, image, texture, quantitative, class_index):
    layer_name = "top_activation"
    grad_model = keras.Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    inputs = [
        tf.convert_to_tensor(image[None], tf.float32),
        tf.convert_to_tensor(texture[None], tf.float32),
        tf.convert_to_tensor(quantitative[None], tf.float32),
    ]
    with tf.GradientTape() as tape:
        conv, pred = grad_model(inputs, training=False)
        score = tf.cast(pred[:, class_index], tf.float32)
    grads = tape.gradient(score, conv)
    if grads is None:
        raise RuntimeError("Grad-CAM gradients are None.")
    conv = tf.cast(conv, tf.float32)
    grads = tf.cast(grads, tf.float32)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.reduce_sum(weights * conv, axis=-1)[0]
    return normalize_heatmap(cam.numpy())


def overlay(image, heatmap):
    heatmap = cv2.resize(np.ascontiguousarray(heatmap, np.float32), (IMG_SIZE, IMG_SIZE))
    color = cv2.applyColorMap(np.uint8(np.clip(heatmap, 0, 1) * 255), cv2.COLORMAP_JET)
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(image.astype(np.uint8), 0.55, color, 0.45, 0)


def modality_ablation(model, image, texture, quantitative):
    full = model.predict([image[None], texture[None], quantitative[None]], verbose=0)[0]
    pred = int(np.argmax(full))
    zeros_t = np.zeros_like(texture)
    zeros_q = np.zeros_like(quantitative)
    zeros_i = np.zeros_like(image)
    variants = {
        "Full model": full[pred],
        "Without image": model.predict([zeros_i[None], texture[None], quantitative[None]], verbose=0)[0][pred],
        "Without texture": model.predict([image[None], zeros_t[None], quantitative[None]], verbose=0)[0][pred],
        "Without quantitative": model.predict([image[None], texture[None], zeros_q[None]], verbose=0)[0][pred],
    }
    return pred, variants


def save_xai(model, test_df, textures, quantitative, class_names, class_to_index):
    grad_dir = XAI_DIR / "gradcam"
    ablation_dir = XAI_DIR / "modality_ablation"
    combined_dir = XAI_DIR / "combined"
    for folder in [grad_dir, ablation_dir, combined_dir]:
        folder.mkdir(parents=True, exist_ok=True)

    rows = []
    for i in range(min(N_XAI, len(test_df))):
        row = test_df.iloc[i]
        image = preprocess_rgb(read_rgb(row.filepath)).astype(np.float32)
        texture = textures[i]
        quant = quantitative[i]
        probs = model.predict([image[None], texture[None], quant[None]], verbose=0)[0]
        pred = int(np.argmax(probs))
        true = class_to_index[row.label]
        stem = f"Patient_{i+1:02d}_true_{class_names[true]}_pred_{class_names[pred]}"

        try:
            heat = gradcam(model, image, texture, quant, pred)
            grad_image = overlay(image, heat)
        except Exception as exc:
            print("Grad-CAM failed:", exc)
            grad_image = image.astype(np.uint8)

        cv2.imwrite(str(grad_dir / f"{stem}.png"), cv2.cvtColor(grad_image, cv2.COLOR_RGB2BGR))

        _, variants = modality_ablation(model, image, texture, quant)
        labels = list(variants.keys())
        values = list(variants.values())
        plt.figure(figsize=(8, 5))
        plt.bar(labels, values)
        plt.ylim(0, 1)
        plt.ylabel("Predicted-class probability", fontweight="bold")
        plt.title("Modality Ablation", fontweight="bold")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        ablation_path = ablation_dir / f"{stem}.png"
        plt.savefig(ablation_path, dpi=250, bbox_inches="tight")
        plt.close()

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(image.astype(np.uint8)); axes[0].set_title("Preprocessed CT", fontweight="bold"); axes[0].axis("off")
        axes[1].imshow(grad_image); axes[1].set_title("Grad-CAM", fontweight="bold"); axes[1].axis("off")
        axes[2].imshow(plt.imread(ablation_path)); axes[2].set_title("Modality Ablation", fontweight="bold"); axes[2].axis("off")
        fig.suptitle(f"True: {class_names[true]} | Predicted: {class_names[pred]} ({probs[pred]:.4f})", fontweight="bold")
        plt.tight_layout(rect=[0, 0, 1, 0.92])
        plt.savefig(combined_dir / f"{stem}.png", dpi=250, bbox_inches="tight")
        plt.close()

        rows.append({
            "patient_number": i + 1,
            "filepath": row.filepath,
            "true_class": class_names[true],
            "predicted_class": class_names[pred],
            "confidence": float(probs[pred]),
            "correct": int(true == pred),
            **{f"prob_{name}": float(probs[j]) for j, name in enumerate(class_names)},
        })
    pd.DataFrame(rows).to_csv(XAI_DIR / "first_20_xai_predictions.csv", index=False)

# -----------------------------------------------------------------------------
# 12. ZIP
# -----------------------------------------------------------------------------
def zip_results():
    path = WORK_DIR / "TriLiver_FuseNet_Complete_Results.zip"
    if path.exists():
        path.unlink()
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as archive:
        for file in RESULTS_DIR.rglob("*"):
            if file.is_file():
                archive.write(file, arcname=file.relative_to(RESULTS_DIR.parent))
    return path


def automatic_download(path):
    if in_colab():
        from google.colab import files
        files.download(str(path))
    else:
        print("Results ZIP:", path.resolve())

# -----------------------------------------------------------------------------
# 13. MAIN
# -----------------------------------------------------------------------------
def main():
    pipeline_start = time.perf_counter()
    slugs = download_data()
    (RESULTS_DIR / "downloaded_kaggle_sources.txt").write_text("\n".join(slugs), encoding="utf-8")

    all_df = collect_images(DATA_DIR)
    class_names = [name for name in TARGET_CLASS_ORDER if name in all_df.label.unique()]
    class_to_index = {name: i for i, name in enumerate(class_names)}
    (RESULTS_DIR / "class_mapping.json").write_text(json.dumps(class_to_index, indent=2), encoding="utf-8")

    train_df, val_df, test_df = create_splits(all_df)
    save_preprocessing_samples(train_df)

    feature_start = time.perf_counter()
    train_texture, train_quant = extract_feature_arrays(train_df)
    val_texture, val_quant = extract_feature_arrays(val_df)
    test_texture, test_quant = extract_feature_arrays(test_df)

    texture_scaler = StandardScaler().fit(train_texture)
    quant_scaler = StandardScaler().fit(train_quant)
    train_texture = texture_scaler.transform(train_texture).astype(np.float32)
    val_texture = texture_scaler.transform(val_texture).astype(np.float32)
    test_texture = texture_scaler.transform(test_texture).astype(np.float32)
    train_quant = quant_scaler.transform(train_quant).astype(np.float32)
    val_quant = quant_scaler.transform(val_quant).astype(np.float32)
    test_quant = quant_scaler.transform(test_quant).astype(np.float32)
    feature_time = time.perf_counter() - feature_start

    import joblib
    joblib.dump(texture_scaler, MODEL_DIR / "texture_scaler.joblib")
    joblib.dump(quant_scaler, MODEL_DIR / "quantitative_scaler.joblib")

    train_ds = make_dataset(train_df, train_texture, train_quant, class_to_index, training=True)
    train_eval_ds = make_dataset(train_df, train_texture, train_quant, class_to_index, training=False)
    val_ds = make_dataset(val_df, val_texture, val_quant, class_to_index, training=False)
    test_ds = make_dataset(test_df, test_texture, test_quant, class_to_index, training=False)

    model, backbone = build_triliver_fusenet(len(class_names), train_texture.shape[1], train_quant.shape[1])
    compile_model(model, INITIAL_LR)
    with open(MODEL_DIR / "model_summary.txt", "w", encoding="utf-8") as handle:
        model.summary(print_fn=lambda line: handle.write(line + "\n"))

    y_train = train_df.label.map(class_to_index).values
    weights = compute_class_weight(class_weight="balanced", classes=np.arange(len(class_names)), y=y_train)
    class_weights = {i: float(w) for i, w in enumerate(weights)}
    print("Class weights:", class_weights)

    best_path = MODEL_DIR / "best_triliver_fusenet.weights.h5"
    callbacks1 = [
        keras.callbacks.ModelCheckpoint(str(best_path), monitor="val_loss", mode="min", save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=4, min_lr=1e-7, verbose=1),
        keras.callbacks.CSVLogger(str(METRICS_DIR / "head_training_log.csv")),
    ]

    training_start = time.perf_counter()
    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks1,
        verbose=1,
    )

    unfreeze_backbone(backbone, 60)
    compile_model(model, FINE_TUNE_LR)
    callbacks2 = [
        keras.callbacks.ModelCheckpoint(str(best_path), monitor="val_loss", mode="min", save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=4, min_lr=1e-8, verbose=1),
        keras.callbacks.CSVLogger(str(METRICS_DIR / "fine_tuning_log.csv")),
    ]
    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=HEAD_EPOCHS,
        epochs=TOTAL_EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks2,
        verbose=1,
    )
    training_time = time.perf_counter() - training_start

    # Reload only the best weights into the already-built architecture. This
    # avoids Keras deserialization issues and preserves the exact model graph.
    if best_path.exists():
        model.load_weights(best_path)
        print("Loaded best checkpoint weights:", best_path)

    # The architecture now contains no Lambda layers, so the final model is
    # safely serializable and can be loaded later with keras.models.load_model.
    model.save(MODEL_DIR / "final_triliver_fusenet.keras")
    model.save_weights(MODEL_DIR / "final_triliver_fusenet.weights.h5")

    history = merge_histories(h1, h2)
    save_curves(history)

    all_metrics = []
    for split, ds in [("training", train_eval_ds), ("validation", val_ds), ("test", test_ds)]:
        overall, y_true, y_pred, probs, per_class = evaluate_split(model, ds, split, class_names)
        all_metrics.append(overall)
        save_confusion(y_true, y_pred, class_names, split)
        save_roc(y_true, probs, class_names, split)
        save_boxplot(per_class, split)

    pd.DataFrame(all_metrics).to_csv(METRICS_DIR / "all_split_overall_metrics.csv", index=False)
    print("\nFINAL METRICS\n", pd.DataFrame(all_metrics).to_string(index=False))

    complexity = count_params(model)
    flops, gflops = estimate_flops(model)
    complexity.update({
        "flops_per_forward_pass": flops,
        "gflops_per_forward_pass": gflops,
        "texture_feature_dimension": int(train_texture.shape[1]),
        "quantitative_feature_dimension": int(train_quant.shape[1]),
        "feature_extraction_time_seconds": float(feature_time),
        "training_time_seconds": float(training_time),
        "training_time_minutes": float(training_time / 60.0),
        "pipeline_time_before_xai_seconds": float(time.perf_counter() - pipeline_start),
    })
    with open(METRICS_DIR / "model_complexity_and_time.json", "w", encoding="utf-8") as handle:
        json.dump(complexity, handle, indent=2)
    pd.DataFrame([complexity]).to_csv(METRICS_DIR / "model_complexity_and_time.csv", index=False)

    save_xai(model, test_df, test_texture, test_quant, class_names, class_to_index)

    metadata = {
        "model": "TriLiver-FuseNet",
        "modalities": [
            "EfficientNetV2-S global CT deep features",
            "LBP-intensity-edge texture features",
            "GLCM-statistical quantitative features",
        ],
        "novel_modules": [
            "adaptive modality attention",
            "gated residual fusion",
            "multi-representation feature learning",
            "modality-ablation explainability",
        ],
        "classes": class_names,
        "kaggle_sources": slugs,
    }
    (RESULTS_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    zip_path = zip_results()
    print("\nCompleted successfully.")
    print("ZIP:", zip_path)
    automatic_download(zip_path)


if __name__ == "__main__":
    main()



TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision: <DTypePolicy "mixed_float16">

Please upload kaggle.json...


Saving kaggle.json to kaggle.json
[CMD] kaggle kernels pull ahmedhamza1996/liver-tumor-classification -p /content/liver_triliver_fusenet_work/kaggle_kernel_metadata -m
Source code and metadata downloaded to /content/liver_triliver_fusenet_work/kaggle_kernel_metadata

Discovered Kaggle dataset sources: ['ahmedhamza1996/dataset']
[CMD] kaggle datasets download -d ahmedhamza1996/dataset -p /content/liver_triliver_fusenet_work/dataset/ahmedhamza1996__dataset --unzip
Dataset URL: https://www.kaggle.com/datasets/ahmedhamza1996/dataset
License(s): unknown

  0%|          | 0.00/241M [00:00<?, ?B/s]
  3%|▎         | 7.00M/241M [00:00<00:06, 35.9MB/s]
  9%|▊         | 21.0M/241M [00:00<00:02, 78.7MB/s]
 12%|█▏        | 30.0M/241M [00:00<00:03, 61.0MB/s]
 17%|█▋        | 41.0M/241M [00:00<00:03, 63.5MB/s]
 22%|██▏       | 53.0M/241M [00:00<00:02, 68.2MB/s]
 27%|██▋       | 65.0M/241M [00:01<00:02, 67.7MB/s]
 33%|███▎      | 79.0M/241M [00:01<00:02, 82.3MB/s]
 37%|███▋      | 88.0M/241M [00:01<00

Class weights: {0: 2.5108225108225106, 1: 0.5101143359718557, 2: 1.5591397849462365}
Epoch 1/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5909 - loss: 0.8044   
Epoch 1: val_loss improved from None to 0.27589, saving model to /content/liver_triliver_fusenet_work/results_triliver_fusenet/model/best_triliver_fusenet.weights.h5

Epoch 1: finished saving model to /content/liver_triliver_fusenet_work/results_triliver_fusenet/model/best_triliver_fusenet.weights.h5
37/37 ━━━━━━━━━━━━━━━━━━━━ 250s 4s/step - accuracy: 0.7207 - loss: 0.5287 - val_accuracy: 0.8904 - val_loss: 0.2759 - learning_rate: 0.0010
Epoch 2/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9285 - loss: 0.1787
Epoch 2: val_loss improved from 0.27589 to 0.02264, saving model to /content/liver_triliver_fusenet_work/results_triliver_fusenet/model/best_triliver_fusenet.weights.h5

Epoch 2: finished saving model to /content/liver_triliver_fusenet_work/results_triliver_fusenet/model/best_triliver_fusenet.weights.h5

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.



Completed successfully.
ZIP: /content/liver_triliver_fusenet_work/TriLiver_FuseNet_Complete_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>